# C3S seasonal forecasts: quickstart

A minimal end-to-end pull of C3S seasonal forecast data from the Copernicus
Climate Data Store, opened with xarray, and plotted to show ensemble spread
across lead times.

C3S seasonal forecasts are probabilistic predictions 1 to 6 months ahead,
produced by nine operational centres. Each centre runs an ensemble of
coupled atmosphere-ocean model simulations. The spread across ensemble
members quantifies forecast uncertainty. This notebook uses the monthly
statistics product (`seasonal-monthly-single-levels`), which provides
monthly means per ensemble member on a 1 x 1 degree grid.

Full reference: [`docs/c3s-seasonal/README.md`](../docs/c3s-seasonal/README.md).

## Before you run this

1. Register for a free CDS account at https://cds.climate.copernicus.eu/
2. Accept **both** licences on the dataset page: the standard "Licence to
   use Copernicus Products" and the additional licence for non-European
   seasonal forecast centres.
3. Copy your Personal Access Token from the CDS profile page and save it to
   `~/.cdsapirc` (see the ERA5 docs for the exact format).
4. Install the Python stack:
   ```bash
   pip install cdsapi xarray cfgrib eccodes matplotlib cartopy numpy
   ```

The default config below pulls ECMWF system 51 monthly-mean 2 metre
temperature for a January 2024 initialisation, all six lead times, over
Europe. Output is GRIB format (about 2-5 MB). Requests may take a few
minutes due to tape archive access.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
# CDS API variable name. See the CDS dataset page for the full catalogue.
VARIABLE = "2m_temperature"

# Originating centre and system version. System numbers are centre-specific.
ORIGINATING_CENTRE = "ecmwf"
SYSTEM = "51"

# Product type: "monthly_mean" gives one value per ensemble member per month.
PRODUCT_TYPE = "monthly_mean"

# Initialisation date. Seasonal forecasts are issued monthly.
YEAR = "2024"
MONTH = "01"

# Lead times in months ahead (1 = first full calendar month after init).
LEADTIME_MONTHS = ["1", "2", "3", "4", "5", "6"]

# Bounding box as [north, west, south, east]. Default: Europe.
AREA = [72, -12, 35, 30]

# GRIB is strongly recommended. NetCDF is experimental for this dataset.
FORMAT = "grib"

# Where to save the output, relative to the notebooks/ directory.
OUTPUT_DIR = "../data/c3s-seasonal"
OUTPUT_FILENAME = "c3s_seasonal_quickstart.grib"
# ==================================================================

## Imports and environment check

The version printout helps reproduce results when notebooks behave
differently on two machines. The credential check fails fast with a clear
error if `~/.cdsapirc` is missing.

This dataset uses GRIB format, so we need `cfgrib` as the xarray engine
instead of the `netcdf4` engine used for ERA5.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import cdsapi
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
for pkg in ["cdsapi", "xarray", "cfgrib", "eccodes", "matplotlib", "cartopy", "numpy"]:
    try:
        print(f"{pkg:12} {version(pkg)}")
    except Exception:
        print(f"{pkg:12} not found")

# Find the repo root by walking up from CWD until we see CLAUDE.md.
# This works whether Jupyter was launched from the repo root or from
# the notebooks/ directory.
def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError(
        "Could not find repo root. Run this notebook from inside the "
        "climate-data-quickstart repository."
    )

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.credentials import check_cds_credentials
check_cds_credentials()
print("\nCDS credentials found.")

## Submit the request

The CDS queues requests server-side. Seasonal forecast data is served from
tape archive and can be slower than ERA5 requests, typically a few minutes
for a small pull like this one.

The `product_type` of `monthly_mean` returns one monthly-mean value per
ensemble member. With ECMWF system 51, that means 51 ensemble members at
each of the 6 lead times. `leadtime_month=1` is the first full calendar
month after the initialisation date: for a January 2024 initialisation,
lead time 1 is February 2024, lead time 2 is March 2024, and so on.

In [ ]:
output_path = Path(OUTPUT_DIR) / OUTPUT_FILENAME
output_path.parent.mkdir(parents=True, exist_ok=True)

request = {
    "format": FORMAT,
    "originating_centre": ORIGINATING_CENTRE,
    "system": SYSTEM,
    "variable": VARIABLE,
    "product_type": PRODUCT_TYPE,
    "year": YEAR,
    "month": MONTH,
    "leadtime_month": LEADTIME_MONTHS,
    "area": AREA,
}

client = cdsapi.Client()
client.retrieve("seasonal-monthly-single-levels", request).download(str(output_path))
print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

## Open and inspect

GRIB files from C3S seasonal forecasts contain multiple "messages", one per
ensemble member per lead time. `cfgrib` may split these across multiple
datasets if the GRIB messages have incompatible structures. We use
`xr.open_datasets()` (plural) with `engine="cfgrib"` to handle this, then
select the first dataset which contains our variable.

Look for the `number` dimension (ensemble member index) and `step` or
`forecastMonth` dimension (lead time). Values are in kelvin.

In [ ]:
# cfgrib may return multiple datasets if GRIB messages have different
# structures. open_datasets returns a list; we pick the one with our variable.
datasets = xr.open_datasets(str(output_path), engine="cfgrib")
print(f"cfgrib returned {len(datasets)} dataset(s)\n")

# Find the dataset containing 2m temperature (GRIB short name: "t2m" or "2t")
ds = None
for d in datasets:
    for var_name in d.data_vars:
        if var_name in ("t2m", "2t"):
            ds = d
            break
    if ds is not None:
        break

if ds is None:
    # Fall back to the first dataset if the expected name is not found
    ds = datasets[0]
    print("Warning: expected variable name not found, using first dataset")

print(ds)

## Plot: ensemble spread across lead times

Seasonal forecasts are probabilistic, not deterministic. The key output is
the distribution across ensemble members, not a single "best guess" value.

The plot below shows two views:

1. **Box plot** of area-mean 2 m temperature (converted to Celsius) across
   all ensemble members at each lead time. The box shows the interquartile
   range, the whiskers extend to 1.5x IQR, and the line marks the median.
   Wider boxes mean more ensemble disagreement (less confident forecast).

2. **Map** of the ensemble mean at lead time 1 (the first full month after
   initialisation). This shows the spatial pattern of the forecast.

In [ ]:
# Identify the temperature variable name (cfgrib uses GRIB short names)
t2m_name = None
for var_name in ds.data_vars:
    if var_name in ("t2m", "2t"):
        t2m_name = var_name
        break
if t2m_name is None:
    t2m_name = list(ds.data_vars)[0]

t2m = ds[t2m_name]

# Identify the lead-time dimension (may be "step", "forecastMonth", or similar)
step_dim = None
for dim in t2m.dims:
    if dim in ("step", "forecastMonth"):
        step_dim = dim
        break
# If neither standard name found, look for a dimension that is not lat/lon/number
if step_dim is None:
    for dim in t2m.dims:
        if dim not in ("latitude", "longitude", "number", "time"):
            step_dim = dim
            break

# Identify the ensemble member dimension
ens_dim = "number" if "number" in t2m.dims else None

# Convert to Celsius
t2m_c = t2m - 273.15

# --- Figure 1: Box plot of area-mean ensemble spread by lead time ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Compute area mean over lat/lon for each member and lead time
spatial_dims = [d for d in t2m_c.dims if d in ("latitude", "longitude")]
t2m_area_mean = t2m_c.mean(dim=spatial_dims)

if step_dim is not None:
    n_steps = t2m_area_mean.sizes[step_dim]
    step_vals = t2m_area_mean[step_dim].values

    # Build data for box plot: one list of member values per lead time
    box_data = []
    labels = []
    for i in range(n_steps):
        sel = t2m_area_mean.isel({step_dim: i})
        if ens_dim is not None:
            vals = sel.values.flatten()
        else:
            vals = np.atleast_1d(sel.values).flatten()
        box_data.append(vals[~np.isnan(vals)])

        # Label as calendar month (init month + lead time)
        if step_dim == "step":
            # step is a timedelta; convert to approximate months
            step_val = step_vals[i]
            if hasattr(step_val, 'days'):
                months_ahead = max(1, int(round(step_val.days / 30.44)))
            elif isinstance(step_val, np.timedelta64):
                days = step_val / np.timedelta64(1, 'D')
                months_ahead = max(1, int(round(days / 30.44)))
            else:
                months_ahead = i + 1
            labels.append(f"LT {months_ahead}")
        else:
            labels.append(f"LT {int(step_vals[i])}")

    bp = axes[0].boxplot(box_data, labels=labels, patch_artist=True)
    for patch in bp["boxes"]:
        patch.set_facecolor("#4C72B0")
        patch.set_alpha(0.7)
else:
    # No step dimension found: single lead time
    if ens_dim is not None:
        vals = t2m_area_mean.values.flatten()
    else:
        vals = np.atleast_1d(t2m_area_mean.values).flatten()
    bp = axes[0].boxplot([vals[~np.isnan(vals)]], labels=["LT 1"], patch_artist=True)
    bp["boxes"][0].set_facecolor("#4C72B0")
    bp["boxes"][0].set_alpha(0.7)

axes[0].set_xlabel("Lead time (months ahead)")
axes[0].set_ylabel("Area-mean 2 m temperature (C)")
axes[0].set_title(f"Ensemble spread, init {YEAR}-{MONTH}, ECMWF sys {SYSTEM}")
axes[0].grid(axis="y", alpha=0.3)

# --- Figure 2: Map of ensemble mean at lead time 1 ---
ax_map = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
# Remove the plain axes created by subplots
axes[1].remove()

# Select lead time 1 and compute ensemble mean
if step_dim is not None:
    t2m_lt1 = t2m_c.isel({step_dim: 0})
else:
    t2m_lt1 = t2m_c

if ens_dim is not None:
    t2m_ens_mean = t2m_lt1.mean(dim=ens_dim)
else:
    t2m_ens_mean = t2m_lt1

t2m_ens_mean.plot(
    ax=ax_map,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    cbar_kwargs={"label": "2 m temperature (C)", "shrink": 0.8},
)

ax_map.coastlines(resolution="50m", linewidth=0.8)
ax_map.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
gl = ax_map.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# Build a title showing the valid month for lead time 1
init_month_int = int(MONTH)
valid_month = init_month_int + 1 if init_month_int < 12 else 1
valid_year = int(YEAR) if init_month_int < 12 else int(YEAR) + 1
ax_map.set_title(f"Ensemble mean, {valid_year}-{valid_month:02d} (LT 1)")

plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/c3s-seasonal/README.md`](../docs/c3s-seasonal/README.md)
  for variables, contributing centres, licence, citation, and gotchas.
- **Compare centres**: change `ORIGINATING_CENTRE` and `SYSTEM` to pull
  forecasts from a different centre (e.g., `ukmo` / `610` for the UK Met
  Office). Comparing ensemble spreads across centres shows where models
  agree and disagree.
- **Hindcasts for bias correction**: request `product_type="hindcast_climate_mean"`
  or individual hindcast members for the 1993-2016 period. Hindcasts are
  essential for calibrating real-time forecasts against observed climatology.
- **Other variables**: try `total_precipitation` or
  `mean_sea_level_pressure` for a different view of the seasonal outlook.
- **Anomaly maps**: subtract the ensemble-mean hindcast climatology from the
  real-time forecast to produce anomaly maps, which are easier to interpret
  than raw values.
- **Related datasets in this repo**: ERA5 single levels for the historical
  record, CMIP6 for multi-decade projections, ECMWF Open Data for
  short-range operational forecasts (days ahead).